# DDRI Full-Station Station-Hour LightGBM Objective Comparison

161개 공통 스테이션 `station-hour` 데이터셋을 사용해 전체 서비스용 LightGBM objective를 비교한다.

- Train: `2023`
- Validation: `2024`
- Final Train: `2023 + 2024`
- Test: `2025`

비교 대상:

- `LightGBM_RMSE_Full`
- `LightGBM_Poisson_Full`

전체 스테이션 본실험에서는 먼저 계산 효율이 좋은 LightGBM 계열에서 objective를 확정하고, CatBoost는 필요 시 별도 축소 실험으로 분리한다.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
ROOT = Path('/Users/cheng80/Desktop/ddri_work')
SOURCE_DIR = ROOT / '3조 공유폴더/군집별 데이터_전체 스테이션/full_data'
WORK_DIR = ROOT / 'works/06_prediction_long_full'
OUTPUT_DATA_DIR = WORK_DIR / 'output/data'
IMAGE_DIR = WORK_DIR / 'output/images'
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = SOURCE_DIR / 'ddri_prediction_long_train_2023_2024.csv'
TEST_PATH = SOURCE_DIR / 'ddri_prediction_long_test_2025.csv'

DTYPE_MAP = {
    'station_id': 'int32',
    'hour': 'int8',
    'rental_count': 'int16',
    'cluster': 'int8',
    'mapped_dong_code': 'int32',
    'weekday': 'int8',
    'month': 'int8',
    'holiday': 'int8',
    'temperature': 'float32',
    'humidity': 'float32',
    'precipitation': 'float32',
    'wind_speed': 'float32',
}

In [3]:
train = pd.read_csv(TRAIN_PATH, dtype=DTYPE_MAP)
test = pd.read_csv(TEST_PATH, dtype=DTYPE_MAP)

for df in [train, test]:
    df['date'] = pd.to_datetime(df['date'])
    df['datetime'] = df['date'] + pd.to_timedelta(df['hour'].astype('int16'), unit='h')

combined = pd.concat([train.assign(dataset='train'), test.assign(dataset='test')], ignore_index=True)
combined = combined.sort_values(['station_id', 'datetime']).reset_index(drop=True)

grouped_target = combined.groupby('station_id')['rental_count']
combined['lag_1h'] = grouped_target.shift(1).astype('float32')
combined['lag_24h'] = grouped_target.shift(24).astype('float32')
combined['lag_168h'] = grouped_target.shift(168).astype('float32')
combined['rolling_mean_24h'] = grouped_target.transform(lambda s: s.shift(1).rolling(24).mean()).astype('float32')
combined['rolling_mean_168h'] = grouped_target.transform(lambda s: s.shift(1).rolling(168).mean()).astype('float32')
combined['rolling_std_24h'] = grouped_target.transform(lambda s: s.shift(1).rolling(24).std()).astype('float32')

train_model = combined[combined['dataset'] == 'train'].copy()
test_model = combined[combined['dataset'] == 'test'].copy()

FEATURE_COLUMNS = [
    'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_mean_168h', 'rolling_std_24h'
]
CATEGORICAL_COLUMNS = ['station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday']

train_2023 = train_model[train_model['date'] < '2024-01-01'].copy()
valid_2024 = train_model[train_model['date'] >= '2024-01-01'].copy()
full_train = train_model.copy()

X_train = train_2023[FEATURE_COLUMNS].copy()
y_train = train_2023['rental_count'].copy()
X_valid = valid_2024[FEATURE_COLUMNS].copy()
y_valid = valid_2024['rental_count'].copy()
X_full = full_train[FEATURE_COLUMNS].copy()
y_full = full_train['rental_count'].copy()
X_test = test_model[FEATURE_COLUMNS].copy()
y_test = test_model['rental_count'].copy()

for frame in [X_train, X_valid, X_full, X_test]:
    for column in CATEGORICAL_COLUMNS:
        frame[column] = frame[column].astype('category')

train_2023.shape, valid_2024.shape, full_train.shape, test_model.shape

((1410360, 21), (1414224, 21), (2824584, 21), (1410360, 21))

In [4]:
def evaluate_predictions(model_name: str, split_name: str, y_true: pd.Series, y_pred) -> dict:
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {
        'model': model_name,
        'split': split_name,
        'rmse': round(float(rmse), 4),
        'mae': round(float(mae), 4),
        'r2': round(float(r2), 4),
    }

results = []

lightgbm_rmse = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='regression',
)
lightgbm_rmse.fit(X_train, y_train, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM_RMSE_Full', 'validation_2024', y_valid, lightgbm_rmse.predict(X_valid)))
lightgbm_rmse.fit(X_full, y_full, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM_RMSE_Full', 'test_2025_refit', y_test, lightgbm_rmse.predict(X_test)))

lightgbm_poisson = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='poisson',
)
lightgbm_poisson.fit(X_train, y_train, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM_Poisson_Full', 'validation_2024', y_valid, lightgbm_poisson.predict(X_valid)))
lightgbm_poisson.fit(X_full, y_full, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM_Poisson_Full', 'test_2025_refit', y_test, lightgbm_poisson.predict(X_test)))

results_df = pd.DataFrame(results)
results_df

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.032717 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1618
[LightGBM] [Info] Number of data points in the train set: 1410360, number of used features: 17
[LightGBM] [Info] Start training from score 0.688392


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.064176 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1642
[LightGBM] [Info] Number of data points in the train set: 2824584, number of used features: 17
[LightGBM] [Info] Start training from score 0.693763


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.032123 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1618
[LightGBM] [Info] Number of data points in the train set: 1410360, number of used features: 17
[LightGBM] [Info] Start training from score -0.373396


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031630 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1642
[LightGBM] [Info] Number of data points in the train set: 2824584, number of used features: 17
[LightGBM] [Info] Start training from score -0.365625


,model,split,rmse,mae,r2
0,LightGBM_RMSE_Full,validation_2024,0.9735,0.6234,0.4463
1,LightGBM_RMSE_Full,test_2025_refit,0.8624,0.5594,0.4369
2,LightGBM_Poisson_Full,validation_2024,0.9827,0.6260,0.4359
3,LightGBM_Poisson_Full,test_2025_refit,0.8704,0.5613,0.4262


In [5]:
metrics_path = OUTPUT_DATA_DIR / 'ddri_station_hour_full_model_comparison_metrics.csv'
results_df.to_csv(metrics_path, index=False, encoding='utf-8-sig')

best_model = lightgbm_rmse
importance_df = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance': best_model.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)
importance_path = OUTPUT_DATA_DIR / 'ddri_station_hour_full_model_comparison_lightgbm_feature_importance.csv'
importance_df.to_csv(importance_path, index=False, encoding='utf-8-sig')

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(12), x='importance', y='feature', hue='feature', legend=False, palette='Blues_r')
plt.title('DDRI Full-Station LightGBM Objective Comparison Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
feature_image_path = IMAGE_DIR / 'ddri_station_hour_full_model_comparison_lightgbm_feature_importance.png'
plt.savefig(feature_image_path, dpi=150)
plt.close()

metrics_path, importance_path, feature_image_path

(PosixPath('/Users/cheng80/Desktop/ddri_work/works/06_prediction_long_full/output/data/ddri_station_hour_full_model_comparison_metrics.csv'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/works/06_prediction_long_full/output/data/ddri_station_hour_full_model_comparison_lightgbm_feature_importance.csv'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/works/06_prediction_long_full/output/images/ddri_station_hour_full_model_comparison_lightgbm_feature_importance.png'))

In [6]:
results_df, importance_df.head(12)

(                   model            split    rmse     mae      r2
 0     LightGBM_RMSE_Full  validation_2024  0.9735  0.6234  0.4463
 1     LightGBM_RMSE_Full  test_2025_refit  0.8624  0.5594  0.4369
 2  LightGBM_Poisson_Full  validation_2024  0.9827  0.6260  0.4359
 3  LightGBM_Poisson_Full  test_2025_refit  0.8704  0.5613  0.4262,
               feature  importance
 0          station_id        2297
 1                hour        1692
 2             weekday         563
 3            lag_168h         556
 4              lag_1h         491
 5         temperature         488
 6             lag_24h         414
 7   rolling_mean_168h         331
 8            humidity         314
 9    rolling_mean_24h         305
 10      precipitation         294
 11              month         284)